In [1]:
pip install -q transformers datasets accelerate peft evaluate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.4 MB/s eta 0:00:00


In [ ]:
import torch
import json
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    Trainer, 
    TrainingArguments, 
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import random

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
from huggingface_hub import login
login()

In [4]:
from huggingface_hub import whoami
print(whoami()["auth"]["accessToken"]["role"])   # should print "read"

read


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,               # Store weights in 4-bit → 75 % VRAM saved
    bnb_4bit_quant_type="nf4",        # Normal-float-4: best quality for 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # Mat-mul done in 16-bit for speed
    bnb_4bit_use_double_quant=True,   # Additional quantization for memory savings
)

model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
# Prepare model for k-bit training (IMPORTANT: must be done before LoRA)
base_model = prepare_model_for_kbit_training(base_model)

# Improved LoRA config - targeting more modules for better adaptation
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # More modules
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 1,843,200 || all params: 2,508,015,616 || trainable%: 0.0735


In [ ]:
from datasets import load_dataset

ds = load_dataset("qiaojin/PubMedQA", "pqa_artificial")
label2id = {"yes": 0, "no": 1, "maybe": 2}
id2label = {0: "yes", 1: "no", 2: "maybe"}

def fmt(batch):
    """Format with context for better medical QA performance"""
    question = batch['question']
    context = batch.get('context', '')
    final_decision = batch["final_decision"]
    
    # Use context if available, otherwise just question
    if context and isinstance(context, str) and len(context.strip()) > 0:
        # Truncate context if too long (keep first 500 chars)
        context_truncated = context[:500] + "..." if len(context) > 500 else context
        prompt = f"Context: {context_truncated}\n\nQuestion: {question}\nAnswer:"
    else:
        prompt = f"Question: {question}\nAnswer:"
    
    lbl_int = label2id[final_decision]
    target = " " + id2label[lbl_int]
    return {"text": prompt + target}

train_val = ds["train"].train_test_split(test_size=0.15, seed=42)
train_ds = train_val["train"].select(range(5_000)).map(fmt, remove_columns=ds["train"].column_names)
val_ds   = train_val["test"].select(range(1_000)).map(fmt, remove_columns=ds["train"].column_names)

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
max_len = 512  # Increased for context, still T4-safe with 4-bit quantization

def tok_func(batch):
    """Improved tokenization with better batching"""
    texts = batch["text"] if isinstance(batch["text"], list) else [batch["text"]]

    all_input_ids = []
    all_attention_mask = []
    all_labels = []

    for full_text in texts:
        # Split prompt and answer
        if "Answer:" in full_text:
            prompt = full_text.split("Answer:")[0] + "Answer:"
        else:
            prompt = full_text
            full_text = full_text  # No answer in prompt

        # Tokenize prompt and full text
        prompt_tokens = tokenizer(prompt, truncation=True, max_length=max_len, add_special_tokens=True)
        full_tokens = tokenizer(full_text, truncation=True, max_length=max_len, padding="max_length", add_special_tokens=True)

        # Create labels: mask prompt tokens, keep answer tokens
        labels = full_tokens["input_ids"].copy()
        prompt_len = len(prompt_tokens["input_ids"])
        labels[:prompt_len] = [-100] * prompt_len  # Mask prompt

        all_input_ids.append(full_tokens["input_ids"])
        all_attention_mask.append(full_tokens["attention_mask"])
        all_labels.append(labels)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels
    }

train_tok = train_ds.map(tok_func, batched=True, batch_size=100, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(tok_func, batched=True, batch_size=100, remove_columns=val_ds.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [26]:
train_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_tok.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Data collator for dynamic padding
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
    pad_to_multiple_of=8,  # Optimize for tensor cores
)

args = TrainingArguments(
    output_dir="gemma2-pubmedqa-qlora",
    per_device_train_batch_size=2,      # T4-safe
    per_device_eval_batch_size=4,       # Can be larger for eval
    gradient_accumulation_steps=8,      # effective batch = 16
    num_train_epochs=2,                 # Increased epochs
    learning_rate=2e-4,
    warmup_steps=100,                   # Learning rate warmup
    fp16=True,                          # 16-bit mixed precision
    eval_strategy="steps",
    eval_steps=250,                     # More frequent evaluation
    save_steps=250,
    save_total_limit=3,                 # Keep only last 3 checkpoints
    logging_steps=50,
    max_steps=1000,                     # Increased training steps
    report_to="none",                   # no wandb
    load_best_model_at_end=True,       # Load best model
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",          # Memory-efficient optimizer
    lr_scheduler_type="cosine",         # Cosine learning rate schedule
    seed=42,                            # Reproducibility
    dataloader_pin_memory=True,         # Faster data loading
)

In [28]:
trainer = Trainer(
    model=model,               # QLoRA-wrapped Gemma-2B
    args=args,                 # hyper-params we just built
    train_dataset=train_tok, # 15 k tokenised prompts
    eval_dataset=val_tok,      # 3 k tokenised prompts
    data_collator=data_collator,  # pads batches correctly
)

In [29]:
trainer.train( )

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
500,3.915400,4.149624


TrainOutput(global_step=500, training_loss=3.9797820434570315, metrics={'train_runtime': 2191.991, 'train_samples_per_second': 3.65, 'train_steps_per_second': 0.228, 'total_flos': 3.652750335069389e+16, 'train_loss': 3.9797820434570315, 'epoch': 1.5984})

In [ ]:
# 1. Compute validation loss
eval_results = trainer.evaluate(val_tok)
print(f"Validation loss: {eval_results['eval_loss']:.4f}")

# 2. Improved prediction function with better generation parameters
def predict(question, max_new_tokens=5, include_context=False):
    """Generate prediction with improved parameters"""
    if include_context and "Context:" in question:
        prompt = question.split("Answer:")[0] + "Answer:"
    elif "Question:" in question:
        prompt = question.split("Answer:")[0] + "Answer:"
    else:
        prompt = f"Question: {question}\nAnswer:"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,  # Greedy decoding for deterministic results
            temperature=0.0,  # Disable sampling
            repetition_penalty=1.1,  # Reduce repetition
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 3. Comprehensive evaluation function
def evaluate_model(dataset, num_samples=None, verbose=True):
    """Evaluate model on dataset with proper metrics"""
    if num_samples is None:
        num_samples = len(dataset)
    
    correct = 0
    total = 0
    label_counts = {"yes": 0, "no": 0, "maybe": 0}
    label_correct = {"yes": 0, "no": 0, "maybe": 0}
    
    for i in range(min(num_samples, len(dataset))):
        sample_text = dataset[i]["text"]
        
        # Extract question and true label
        if "Answer:" in sample_text:
            question = sample_text.split("Answer:")[0]
            true_label = sample_text.split("Answer:")[1].strip().lower()
        else:
            continue
        
        # Get prediction
        pred = predict(question)
        pred_label = pred.split("Answer:")[-1].strip().lower() if "Answer:" in pred else pred.strip().lower()
        
        # Normalize labels
        pred_label = pred_label.split()[0] if pred_label else ""
        true_label = true_label.split()[0] if true_label else ""
        
        # Check if correct
        is_correct = pred_label.startswith(true_label) or true_label.startswith(pred_label)
        if is_correct:
            correct += 1
            label_correct[true_label] = label_correct.get(true_label, 0) + 1
        
        total += 1
        label_counts[true_label] = label_counts.get(true_label, 0) + 1
        
        if verbose and i < 5:
            print(f"\nSample {i+1}:")
            print(f"Q: {question[:80]}...")
            print(f"True: {true_label}")
            print(f"Pred: {pred_label}")
            print(f"Correct: {is_correct}")
    
    accuracy = (correct / total) * 100 if total > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"Evaluation Results:")
    print(f"{'='*50}")
    print(f"Total samples: {total}")
    print(f"Correct: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"\nPer-label accuracy:")
    for label in ["yes", "no", "maybe"]:
        if label_counts[label] > 0:
            label_acc = (label_correct[label] / label_counts[label]) * 100
            print(f"  {label}: {label_correct[label]}/{label_counts[label]} = {label_acc:.2f}%")
    
    return accuracy, correct, total

# Test on a few examples first
print("Testing on 5 samples:")
test_samples = val_ds.select(range(5))
for i, sample in enumerate(test_samples):
    question = sample["text"].split("Answer:")[0] if "Answer:" in sample["text"] else sample["text"]
    pred = predict(question)
    print(f"\nSample {i+1}:")
    print(f"Q: {question[:100]}...")
    print(f"Prediction: {pred}")

# Full evaluation on validation set
print("\n" + "="*50)
print("Full Validation Set Evaluation:")
print("="*50)
val_accuracy, val_correct, val_total = evaluate_model(val_ds, num_samples=len(val_ds), verbose=False)

Validation loss: 4.1496

Q: Question: Does the growth rate of large renal masses oppose active surveillance?
Answer: yes...
Prediction: Question: Question: Does the growth rate of large renal masses oppose active surveillance?

Answer: yes
Answer: yes
Explanation: The growth

Q: Question: Do iDH mutation and neuroglial developmental features define clinically distinct subclasse...
Prediction: Question: Question: Do iDH mutation and neuroglial developmental features define clinically distinct subclasses of lower grade diffuse astrocytic glioma?

Answer: yes
Explanation: yes , iDH mutation and

Q: Question: Does stem cell factor expression after renal ischemia promote tubular epithelial survival?...
Prediction: Question: Question: Does stem cell factor expression after renal ischemia promote tubular epithelial survival?

Answer: yes
Explanation: yes , stem cell factor (

Q: Question: Do patients with osteoporosis prefer once weekly to once daily dosing with alendronate?
An...
Prediction:

In [ ]:
# Save the fine-tuned model
print("Saving fine-tuned model...")
model.save_pretrained("./gemma2-pubmedqa-qlora-final")
tokenizer.save_pretrained("./gemma2-pubmedqa-qlora-final")
print("Model saved successfully!")

# Set base model to eval mode for comparison
base_model.eval()

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=2048, out_features=256, bias=False)
            (lora_dropout): Module

In [ ]:
def predict_base(question, max_new_tokens=5):
    """Baseline model prediction with improved parameters"""
    if "Question:" in question:
        prompt = question.split("Answer:")[0] + "Answer:" if "Answer:" in question else question
    else:
        prompt = f"Question: {question}\nAnswer:"
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# Optional: Load saved model for inference
# from peft import PeftModel
# 
# loaded_model = PeftModel.from_pretrained(
#     base_model,
#     "./gemma2-pubmedqa-qlora-final"
# )
# 
# # For inference, you can merge adapters:
# # merged_model = loaded_model.merge_and_unload()


In [ ]:
# Evaluate baseline model
print("Evaluating baseline (untrained) model:")
print("="*50)

baseline_correct = 0
baseline_total = 100

for i in range(baseline_total):
    sample_text = val_ds[i]["text"]
    question = sample_text.split("Answer:")[0].strip() if "Answer:" in sample_text else sample_text
    true_label = sample_text.split("Answer:")[1].strip().lower() if "Answer:" in sample_text else ""

    pred = predict_base(question)
    pred_label = pred.split("Answer:")[-1].strip().lower() if "Answer:" in pred else pred.strip().lower()
    
    # Normalize
    pred_label = pred_label.split()[0] if pred_label else ""
    true_label = true_label.split()[0] if true_label else ""

    if pred_label.startswith(true_label) or true_label.startswith(pred_label):
        baseline_correct += 1

    if i < 5:
        print(f"\nSample {i+1}:")
        print("Q:", question[:80], "...")
        print("True:", true_label)
        print("Pred:", pred_label)

baseline_accuracy = (baseline_correct / baseline_total) * 100
print(f"\nBaseline accuracy: {baseline_correct}/{baseline_total} = {baseline_accuracy:.2f}%")

# Compare with fine-tuned model
print("\n" + "="*50)
print("Comparison Summary:")
print("="*50)
print(f"Baseline model accuracy: {baseline_accuracy:.2f}%")
print(f"Fine-tuned model accuracy: {val_accuracy:.2f}%")
print(f"Improvement: {val_accuracy - baseline_accuracy:.2f} percentage points")


Q: Question: Does the growth rate of large renal masses oppose active surveillance? ...
True: yes
Pred: yes
answer

Q: Question: Do iDH mutation and neuroglial developmental features define clinicall ...
True: yes
Pred: yes
explanation

Q: Question: Does stem cell factor expression after renal ischemia promote tubular  ...
True: yes
Pred: yes
explanation

Q: Question: Do patients with osteoporosis prefer once weekly to once daily dosing  ...
True: yes
Pred: yes
answer

Q: Question: Does autoimmune Th2-mediated dacryoadenitis in MRL/MpJ mice become Th1 ...
True: yes
Pred: yes
explanation

Baseline accuracy: 98/100 = 98%
